In [3]:
from pathlib import Path
import re
import pandas as pd

# Root folder containing participant folders like: D:/04-Raw_fMRI/sub-3025432
raw_fmri_root = Path(r"D:\04-Raw_fMRI")


def visit_sort_key(label: str):
    """Sort visits naturally: t02, t04, t06, t08, ..."""
    match = re.search(r"\d+", label)
    return (int(match.group()) if match else float("inf"), label)


participant_sessions = {}
all_visits = set()

for participant_dir in sorted(raw_fmri_root.glob("sub-*")):
    if not participant_dir.is_dir():
        continue

    # Extract participant number from folder name: sub-3025432 -> 3025432
    participant_id = participant_dir.name.replace("sub-", "")

    visits = {
        ses_dir.name.replace("ses-", "")
        for ses_dir in participant_dir.glob("ses-*")
        if ses_dir.is_dir()
    }

    participant_sessions[participant_id] = visits
    all_visits.update(visits)

visit_columns = sorted(all_visits, key=visit_sort_key)

rows = []
for participant_id, visits in participant_sessions.items():
    row = {"participant": participant_id}
    for visit in visit_columns:
        row[visit] = visit if visit in visits else ""
    rows.append(row)

verification_df = pd.DataFrame(rows, columns=["participant", *visit_columns])
verification_df.sort_values("participant").reset_index(drop=True)

,participant,t00,t02,t04,t06,t08,t10
0,3002498,,,t04,t06,,
1,3025432,,t02,,t06,,
2,3100205,,,t04,,t08,
3,3123186,,t02,t04,,,
4,3149469,,,t04,t06,,
...,...,...,...,...,...,...,...
70,8782079,,,t04,t06,,
71,8840385,t00,t02,,,,
72,8874623,,,,t06,t08,
73,8980899,,,,t06,t08,


In [5]:
## Missing participants or visits
visit_cols = [c for c in verification_df.columns if c != "participant"]

# Count how many visit columns are present (non-empty) for each participant
visit_count_df = verification_df.copy()
visit_count_df["n_visits"] = visit_count_df[visit_cols].ne("").sum(axis=1)

# Participants who do NOT have exactly 2 visits
not_two_visits_df = visit_count_df[visit_count_df["n_visits"] != 2].copy()

total_participants = len(visit_count_df)
not_two_count = len(not_two_visits_df)
not_two_pct = (not_two_count / total_participants * 100) if total_participants else 0

print(
    f"Participants without exactly 2 visits: {not_two_count}/{total_participants} "
    f"({not_two_pct:.1f}%)"
)

not_two_visits_df[["participant", "n_visits", *visit_cols]].sort_values(
    ["n_visits", "participant"]
).reset_index(drop=True)

Participants without exactly 2 visits: 4/75 (5.3%)


,participant,n_visits,t00,t02,t04,t06,t08,t10
0,5237572,1,,,,t06,,
1,5595871,1,,t02,,,,
2,6258913,1,,t02,,,,
3,6714520,1,t00,,,,,


In [2]:
from pathlib import Path
import re
import pandas as pd

# Root raw fMRI folder
ROOT = Path(r"D:\04-Raw_fMRI")

# Required visit-level modality folders that should contain at least one NIfTI file
REQUIRED_MODALITIES = ["anat", "fmap", "func"]


def has_nifti_files(folder: Path) -> bool:
    if not folder.exists() or not folder.is_dir():
        return False
    return any(
        p.is_file() and (p.name.lower().endswith(".nii") or p.name.lower().endswith(".nii.gz"))
        for p in folder.iterdir()
    )


def expected_mat_filename(sub: str, ses: str) -> str:
    # sub like "sub-5329975" -> "5329975"
    # ses like "ses-t02" -> "t02"
    sub_digits = re.sub(r"^sub-", "", sub)
    ses_label = re.sub(r"^ses-", "", ses)
    return f"CIMAQ_{sub_digits}_{ses_label}_4cond.mat"


def find_mat_files_for_participant_visit(ses_dir: Path, sub: str):
    """
    Accept both naming conventions:
    1) CIMAQ_<participant>_<sesLabel>_4cond.mat (e.g., ..._t02_4cond.mat)
    2) CIMAQ_<participant>_<visitCode>_4cond.mat (e.g., ..._V10_4cond.mat)

    We therefore look for any CIMAQ_<participant>_*_4cond.mat in that ses folder.
    """
    sub_digits = re.sub(r"^sub-", "", sub)
    pattern = re.compile(rf"^CIMAQ_{re.escape(sub_digits)}_.+_4cond\.mat$", re.IGNORECASE)
    return sorted([p.name for p in ses_dir.iterdir() if p.is_file() and pattern.match(p.name)])


rows = []

if not ROOT.exists():
    raise FileNotFoundError(f"Root path does not exist: {ROOT}")

participants = sorted([p for p in ROOT.iterdir() if p.is_dir() and p.name.startswith("sub-")], key=lambda x: x.name)

for sub_dir in participants:
    sub = sub_dir.name
    ses_dirs = sorted([s for s in sub_dir.iterdir() if s.is_dir() and s.name.startswith("ses-")], key=lambda x: x.name)

    for ses_dir in ses_dirs:
        ses = ses_dir.name
        missing = []

        # Check required modality folders and NIfTI content
        for modality in REQUIRED_MODALITIES:
            mod_dir = ses_dir / modality
            if not mod_dir.exists() or not mod_dir.is_dir():
                missing.append(f"missing_folder:{modality}")
            elif not has_nifti_files(mod_dir):
                missing.append(f"missing_nifti_in:{modality}")

        # Check MAT file(s) allowing both tXX and VXX naming conventions
        expected_mat = expected_mat_filename(sub, ses)
        mat_candidates = find_mat_files_for_participant_visit(ses_dir, sub)
        if len(mat_candidates) == 0:
            missing.append(f"missing_mat:any_CIMAQ_<participant>_*_4cond.mat (expected example: {expected_mat})")

        # Check derivatives status
        derivatives_dir = ses_dir / "derivatives"
        if not derivatives_dir.exists() or not derivatives_dir.is_dir():
            derivatives_status = "missing"
        else:
            has_any = any(derivatives_dir.iterdir())
            derivatives_status = "non-empty" if has_any else "empty"

        rows.append(
            {
                "sub": sub,
                "ses": ses,
                "missing": missing,
                "n_missing": len(missing),
                "mat_files_found": mat_candidates,
                "derivatives_status": derivatives_status,
            }
        )

report_df = pd.DataFrame(rows)

print(f"Total participants: {report_df['sub'].nunique()}")
print(f"Total visits: {len(report_df)}")

# Visits with any missing required item
missing_df = report_df[report_df["n_missing"] > 0].copy()
print(f"Visits with missing required items: {len(missing_df)}")

if len(missing_df) == 0:
    print("No missing required items found for anat/fmap/func NIfTI checks and MAT files.")
else:
    display(missing_df[["sub", "ses", "missing", "n_missing", "mat_files_found"]].sort_values(["sub", "ses"]).reset_index(drop=True))

print("\nDerivatives status counts:")
print(report_df["derivatives_status"].value_counts(dropna=False))

# Optional: per-participant summary of missing info
participant_summary = (
    missing_df.groupby("sub", as_index=False)
    .agg(
        n_visits_with_missing=("ses", "count"),
        visits=("ses", lambda x: sorted(x.tolist())),
    )
    .sort_values("sub")
)

print("\nParticipants with at least one missing required item:", len(participant_summary))
if len(participant_summary) > 0:
    display(participant_summary)

# Machine-readable output
machine_readable = report_df[["sub", "ses", "missing", "mat_files_found", "derivatives_status"]].sort_values(["sub", "ses"]).to_dict(orient="records")
print("\nMachine-readable JSON-like records:")
print(machine_readable)


Total participants: 75
Total visits: 146
Visits with missing required items: 17


,sub,ses,missing,n_missing,mat_files_found
0,sub-3291977,ses-t08,[missing_mat:any_CIMAQ_<participant>_*_4cond.m...,1,[]
1,sub-3388201,ses-t04,[missing_mat:any_CIMAQ_<participant>_*_4cond.m...,1,[]
2,sub-3634853,ses-t00,[missing_mat:any_CIMAQ_<participant>_*_4cond.m...,1,[]
3,sub-4220920,ses-t02,[missing_mat:any_CIMAQ_<participant>_*_4cond.m...,1,[]
4,sub-4273359,ses-t08,"[missing_nifti_in:fmap, missing_nifti_in:func,...",3,[]
5,sub-4292802,ses-t10,[missing_mat:any_CIMAQ_<participant>_*_4cond.m...,1,[]
6,sub-4752214,ses-t02,[missing_mat:any_CIMAQ_<participant>_*_4cond.m...,1,[]
7,sub-4867946,ses-t02,"[missing_nifti_in:fmap, missing_nifti_in:func,...",3,[]
8,sub-5213989,ses-t02,[missing_mat:any_CIMAQ_<participant>_*_4cond.m...,1,[]
9,sub-5585902,ses-t08,[missing_mat:any_CIMAQ_<participant>_*_4cond.m...,1,[]



Derivatives status counts:
derivatives_status
empty    146
Name: count, dtype: int64

Participants with at least one missing required item: 17


,sub,n_visits_with_missing,visits
0,sub-3291977,1,[ses-t08]
1,sub-3388201,1,[ses-t04]
2,sub-3634853,1,[ses-t00]
3,sub-4220920,1,[ses-t02]
4,sub-4273359,1,[ses-t08]
5,sub-4292802,1,[ses-t10]
6,sub-4752214,1,[ses-t02]
7,sub-4867946,1,[ses-t02]
8,sub-5213989,1,[ses-t02]
9,sub-5585902,1,[ses-t08]



Machine-readable JSON-like records:
[{'sub': 'sub-3002498', 'ses': 'ses-t04', 'missing': [], 'mat_files_found': ['CIMAQ_3002498_t04_4cond.mat'], 'derivatives_status': 'empty'}, {'sub': 'sub-3002498', 'ses': 'ses-t06', 'missing': [], 'mat_files_found': ['CIMAQ_3002498_t06_4cond.mat'], 'derivatives_status': 'empty'}, {'sub': 'sub-3025432', 'ses': 'ses-t02', 'missing': [], 'mat_files_found': ['CIMAQ_3025432_V10_4cond.mat'], 'derivatives_status': 'empty'}, {'sub': 'sub-3025432', 'ses': 'ses-t06', 'missing': [], 'mat_files_found': ['CIMAQ_3025432_t06_4cond.mat'], 'derivatives_status': 'empty'}, {'sub': 'sub-3100205', 'ses': 'ses-t04', 'missing': [], 'mat_files_found': ['CIMAQ_3100205_t04_4cond.mat'], 'derivatives_status': 'empty'}, {'sub': 'sub-3100205', 'ses': 'ses-t08', 'missing': [], 'mat_files_found': ['CIMAQ_3100205_t08_4cond.mat'], 'derivatives_status': 'empty'}, {'sub': 'sub-3123186', 'ses': 'ses-t02', 'missing': [], 'mat_files_found': ['CIMAQ_3123186_t02_4cond.mat'], 'derivatives_s